# Step 2 — Graft FP16 MTP Head into W8A16 Checkpoint

In [ ]:
# ── Paths 

import os, json
from collections import defaultdict
import torch
from safetensors import safe_open
from safetensors.torch import save_file

W8A16_DIR = "/home/prashant.takale/icml/4_models/qwen3.5-4b-w8a16-clean"   # Step 1 output
BASE_DIR  = "/home/prashant.takale/icml/4_models/qwen-weights"             # original FP16

# Step 1 saves a single file; locate it
W8A16_SF  = os.path.join(W8A16_DIR, "model.safetensors")
BASE_IDX  = os.path.join(BASE_DIR,  "model.safetensors.index.json")

assert os.path.exists(W8A16_SF), f"Step 1 output not found: {W8A16_SF}"
assert os.path.exists(BASE_IDX), f"Base index not found: {BASE_IDX}"
print("Paths OK.", flush=True)

In [ ]:
# ── Load W8A16 tensors 

print("Loading W8A16 tensors ...", flush=True)
tensors, metadata = {}, None
with safe_open(W8A16_SF, "pt") as f:
    metadata = f.metadata()
    for k in f.keys():
        tensors[k] = f.get_tensor(k)
print(f"  Loaded {len(tensors)} W8A16 tensors", flush=True)

In [ ]:
# ── Locate MTP tensors in base sharded checkpoint 

wm = json.load(open(BASE_IDX))["weight_map"]
mtp_keys = [k for k in wm if "mtp" in k.lower()]
print(f"Found {len(mtp_keys)} MTP tensor keys in base:", flush=True)
for k in sorted(mtp_keys):
    print(f"  {k}")

In [ ]:
# ── Graft MTP tensors (FP16) into W8A16 checkpoint 

by_file = defaultdict(list)
for k in mtp_keys:
    by_file[wm[k]].append(k)

for fname, keys in by_file.items():
    with safe_open(os.path.join(BASE_DIR, fname), "pt") as f:
        for k in keys:
            t = f.get_tensor(k)
            # keep bfloat16 as float16 (both are FP16-class; vLLM accepts either)
            tensors[k] = t.to(torch.float16) if t.dtype == torch.bfloat16 else t
            print(f"  + {k}  shape={tuple(t.shape)}  dtype={tensors[k].dtype}")

print(f"\nTotal tensors after graft: {len(tensors)}", flush=True)

In [ ]:
# ── Save merged checkpoint (overwrites Step 1 file in-place) 

print(f"Saving merged checkpoint to {W8A16_SF} ...", flush=True)
save_file(tensors, W8A16_SF, metadata=metadata or {"format": "pt"})
print(f"\n step 2 DONE — MTP head grafted (FP16) into {W8A16_DIR}", flush=True)

In [ ]:
# ── Verify
    
with safe_open(W8A16_SF, "pt") as f:
    all_keys = list(f.keys())

mtp_found = [k for k in all_keys if k.startswith("mtp.")]
main_packed = [k for k in all_keys if "language_model" in k and k.endswith(".weight_packed")]

print(f"Main body W8A16 packed layers : {len(main_packed)}")
print(f"MTP tensors present           : {len(mtp_found)}")
print("MTP keys:", sorted(mtp_found))